In [1]:
import json
from pathlib import Path
import faiss
import numpy as np
import math
import pandas as pd
from sentence_transformers import SentenceTransformer

In [2]:
#ucitavanje podataka


In [3]:
PROJECT_ROOT = Path.cwd()

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_preprocessed.jsonl"
)
TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "train.jsonl"
)
VALIDATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "validation.jsonl"
)
TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "test.jsonl"
)

In [4]:
def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                records.append(record)

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Neispravan JSON u redu {line_number}: {path}"
                ) from error

    return records

In [5]:
chunks = load_jsonl(CHUNKS_PATH)

train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

In [6]:
#provera 
print(f"Broj chunkova: {len(chunks)}")
print(f"Training pitanja: {len(train_data)}")
print(f"Validation pitanja: {len(validation_data)}")
print(f"Test pitanja: {len(test_data)}")

Broj chunkova: 344
Training pitanja: 100
Validation pitanja: 21
Test pitanja: 22


In [7]:
#ucitavanje modela

In [8]:
RETRIEVER_MODEL_NAME = "intfloat/multilingual-e5-base"

retriever_model = SentenceTransformer(
    RETRIEVER_MODEL_NAME
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
#provera
test_embedding = retriever_model.encode(
    ["query: Zašto je bitan kvalitet softvera?"],
    normalize_embeddings=True
)

print(test_embedding.shape)

(1, 768)


In [10]:
passages = [
    "passage: " + chunk["processed_text"]
    for chunk in chunks
]

print(f"Pripremljeno tekstova: {len(passages)}")

Pripremljeno tekstova: 344


In [11]:
chunk_embeddings = retriever_model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [12]:
#provera
print(f"Oblik matrice embeddinga: {chunk_embeddings.shape}")
print(f"Tip podataka: {chunk_embeddings.dtype}")

Oblik matrice embeddinga: (344, 768)
Tip podataka: float32


In [13]:
chunk_embeddings = chunk_embeddings.astype("float32")

embedding_dimension = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(chunk_embeddings)

In [14]:
#provera
print(f"Broj vektora u indeksu: {faiss_index.ntotal}")

Broj vektora u indeksu: 344


In [15]:
def retrieve(
    question: str,
    top_k: int = 5
) -> list[dict]:

    if not isinstance(question, str) or not question.strip():
        raise ValueError("Pitanje ne sme biti prazno.")

    query = "query: " + question.strip()

    query_embedding = retriever_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    number_of_results = min(
        top_k,
        len(chunks)
    )

    scores, indices = faiss_index.search(
        query_embedding,
        number_of_results
    )

    results = []

    for rank, (score, chunk_index) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        chunk = chunks[int(chunk_index)]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "score": float(score),
            "text": chunk["processed_text"],
            "pdf_page_start": chunk["pdf_page_start"],
            "pdf_page_end": chunk["pdf_page_end"],
            "section_ref": chunk.get("section_ref"),
            "heading": chunk.get("heading")
        })

    return results

In [16]:
#provera na jednom primeru
QUESTION_ID = 61

example = next(
    example
    for example in test_data
    if example["id"] == QUESTION_ID
)

question = example["processed_question"]
print("Pitanje:")
print(question)
print("Odgovor:")
print(example["processed_answer"])
retrieved_chunks = retrieve(
    question=question,
    top_k=5
)

Pitanje:
Opisati testove kompatibilnosti pri testiranju softvera.
Odgovor:
Testovi kompatibilnosti proveravaju da softver može pravilno da radi u različitim okruženjima i zajedno sa drugim programima, uređajima ili servisima.


In [17]:
for result in retrieved_chunks:
    print("\n" + "=" * 80)

    print(f'Rang: {result["rank"]}')
    print(f'Chunk: {result["chunk_id"]}')
    print(f'Sličnost: {result["score"]:.4f}')

    print(
        "PDF stranice:",
        result["pdf_page_start"],
        "-",
        result["pdf_page_end"]
    )
    print("\nTekst:")
    print(result["text"][:700])


Rang: 1
Chunk: chunk_0167
Sličnost: 0.8609
PDF stranice: 97 - 98

Tekst:
[Vrste i nivoi testiranja]

čkih podešavanja tokom aktivne sesije — provera se uticaj na postojeći sadržaj korisničkog interfejsa.

U svakom od prethodnih situacija aplikacija bi trebalo da se ponaša stabilno, obezbedi korisniku jasne poruke o greškama i izbegne pad sistema ili gubitak podataka. Cilj ovih provera je otkrivanje potencijalnih grešaka i nepredviđenih ponašanja sistema u realnim, kompleksnim i neobičnim scenarijima koje standardni test slučajevi možda ne pokrivaju.

Testiranje prihvatljivosti

Testovi prihvatljivosti (eng. acceptance testing) treba da omoguće klijentima i korisnicima da se sami uvere da je napravljeni softver u skladu sa njihovim potrebama i očekivanjima. Ovu vr

Rang: 2
Chunk: chunk_0172
Sličnost: 0.8539
PDF stranice: 100 - 100

Tekst:
[Testiranje > 4.3 Tehnike testiranja]

sa krajnjim korisnicima, kako bi se osiguralo da instalacija teče bez grešaka i da sistem nakon instalacije is

In [18]:
def get_relevant_chunk_ids(
    source_pages: list[int],
    chunks: list[dict]
) -> set[str]:

    relevant_ids = {
        chunk["chunk_id"]
        for chunk in chunks
        if any(
            chunk["pdf_page_start"] <= page <= chunk["pdf_page_end"]
            for page in source_pages
        )
    }

    return relevant_ids

In [19]:
def precision_at_k(
    retrieved_ids: list[str],
    relevant_ids: set[str],
    k: int
) -> float:

    top_k_ids = retrieved_ids[:k]

    if not top_k_ids:
        return 0.0

    hits = sum(
        1
        for chunk_id in top_k_ids
        if chunk_id in relevant_ids
    )

    return hits / len(top_k_ids)


def recall_at_k(
    retrieved_ids: list[str],
    relevant_ids: set[str],
    k: int
) -> float:

    if not relevant_ids:
        return 0.0

    top_k_ids = retrieved_ids[:k]

    hits = sum(
        1
        for chunk_id in top_k_ids
        if chunk_id in relevant_ids
    )

    return hits / len(relevant_ids)


def reciprocal_rank(
    retrieved_ids: list[str],
    relevant_ids: set[str]
) -> float:

    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id in relevant_ids:
            return 1.0 / rank

    return 0.0


def dcg_at_k(
    retrieved_ids: list[str],
    relevant_ids: set[str],
    k: int
) -> float:

    return sum(
        1.0 / math.log2(rank + 1)
        for rank, chunk_id in enumerate(retrieved_ids[:k], start=1)
        if chunk_id in relevant_ids
    )


def ndcg_at_k(
    retrieved_ids: list[str],
    relevant_ids: set[str],
    k: int
) -> float:

    ideal_hits = min(len(relevant_ids), k)

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1.0 / math.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg_at_k(retrieved_ids, relevant_ids, k) / idcg

In [23]:
def evaluate_retrieval(
    questions: list[dict],
    chunks: list[dict],
    k_values: list[int]
) -> pd.DataFrame:

    rows = []

    for question in questions:
        relevant_ids = get_relevant_chunk_ids(
            source_pages=question["source_pages"],
            chunks=chunks
        )

        # Preskačemo pitanja za koja u korpusu
        # ne postoji nijedan relevantan chunk.
        if not relevant_ids:
            print(
                f'Preskočeno pitanje ID {question["id"]}: '
                f'nema relevantnih chunkova.'
            )
            continue

        # Preuzimamo kompletan rang da bi MRR
        # koristio stvarnu poziciju prvog relevantnog chunka.
        retrieved = retrieve(
            question=question["processed_question"],
            top_k=len(chunks)
        )

        retrieved_ids = [
            result["chunk_id"]
            for result in retrieved
        ]

        row = {
            "id": question["id"],
            "question": question["processed_question"],
            "num_relevant": len(relevant_ids),
            "mrr": reciprocal_rank(
                retrieved_ids,
                relevant_ids
            )
        }

        for k in k_values:
            top_k_ids = retrieved_ids[:k]

            # Da li je pronađen bar jedan relevantan
            # chunk među prvih k rezultata?
            row[f"hit@{k}"] = float(
                any(
                    chunk_id in relevant_ids
                    for chunk_id in top_k_ids
                )
            )

            row[f"precision@{k}"] = precision_at_k(
                retrieved_ids,
                relevant_ids,
                k
            )

            row[f"recall@{k}"] = recall_at_k(
                retrieved_ids,
                relevant_ids,
                k
            )

            row[f"ndcg@{k}"] = ndcg_at_k(
                retrieved_ids,
                relevant_ids,
                k
            )

        rows.append(row)

    return pd.DataFrame(rows)

In [24]:
K_VALUES = [1, 3, 5, 10]

validation_results = evaluate_retrieval(
    questions=validation_data,
    chunks=chunks,
    k_values=K_VALUES
)

Preskočeno pitanje ID 132: nema relevantnih chunkova.
Preskočeno pitanje ID 141: nema relevantnih chunkova.
Preskočeno pitanje ID 127: nema relevantnih chunkova.
Preskočeno pitanje ID 134: nema relevantnih chunkova.
Preskočeno pitanje ID 136: nema relevantnih chunkova.
Preskočeno pitanje ID 143: nema relevantnih chunkova.
Preskočeno pitanje ID 138: nema relevantnih chunkova.


In [25]:
metric_columns = [
    column
    for column in validation_results.columns
    if column not in {
        "id",
        "question",
        "num_relevant"
    }
]

validation_summary = (
    validation_results[metric_columns]
    .mean()
    .to_frame(name="validation")
)

validation_summary

,validation
mrr,0.676774
hit@1,0.571429
precision@1,0.571429
recall@1,0.104847
ndcg@1,0.571429
hit@3,0.714286
precision@3,0.428571
recall@3,0.235884
ndcg@3,0.466480
hit@5,0.785714
